In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import als

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False, drop_pad=True)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config,
#     X_train,
#     X_val,
#     X_val_seq,
#     ordinal_encoder,
#     return_only_val=False,
#     truncate_val=False,
#     return_just_splitted=True,
# )
pad = ordinal_encoder.unknown_value

CPU times: user 8min 55s, sys: 1min 35s, total: 10min 30s
Wall time: 1min 58s


In [5]:
del X_train, X_val, X_val_seq
gc.collect()

190

In [6]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df[config.col_aid] = ordinal_encoder.transform(
            df[config.col_aid].values.reshape(-1, 1)
        ).astype(int)
df = df[df[config.col_aid] != pad]
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=False)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=False)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

CPU times: user 11min 39s, sys: 1min 15s, total: 12min 55s
Wall time: 3min 6s


0

In [8]:
%%time

model_als = als.train_als_model(data=list_inters, n_items=pad, iterations=50, factors=140, alpha=40.0, use_gpu=True, calculate_training_loss=True)

  0%|          | 0/50 [00:00<?, ?it/s]

CPU times: user 10min 58s, sys: 6.27 s, total: 11min 4s
Wall time: 11min 4s


In [12]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
df_test = df_test.to_pandas()
df_test[config.col_aid] = ordinal_encoder.transform(
    df_test[config.col_aid].values.reshape(-1, 1)
).astype(int)
df_test = df_test[df_test[config.col_aid] != pad]
df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()

recommender = als.AlsRecommender(implicit_model=model_als)
recs = recommender.recommend(data_to_predict, n_recs=config.num_recs)

df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
df_idx["user"] = df_test.to_pandas()["user"].astype(int)
df_idx = pl.from_pandas(df_idx)
recs = recs.join(df_idx, on="index")

recs = recs.to_pandas()
recs['item_id'] = ordinal_encoder.inverse_transform(
        recs["item_id"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
recs = recs.dropna()
recs = pl.from_pandas(recs)

recs.write_parquet(config.cur_model_output_path + f'cur_als_df_test_preds_{config.num_recs}.parquet')

17it [01:32,  5.46s/it]


CPU times: user 4min 1s, sys: 40.5 s, total: 4min 41s
Wall time: 3min 58s


In [13]:
recs.head()

index,item_id,als_score,als_rank,user
i64,i64,f64,i64,i64
0,803544,0.723861,1,13244352
0,152547,0.722554,2,13244352
0,33343,0.704752,3,13244352
0,1562705,0.623137,4,13244352
0,1736857,0.493476,5,13244352


In [11]:
stop

NameError: name 'stop' is not defined

In [ ]:
%%time

recommender = als.AlsRecommender(implicit_model=model_als)
recs = recommender.recommend(val_seq_data, n_recs=config.num_recs)

In [ ]:
df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
df_idx = pl.from_pandas(df_idx)
recs = recs.join(df_idx, on="index")
recs.head()

In [ ]:
%%time

recs = recs.to_pandas()
recs['item_id'] = ordinal_encoder.inverse_transform(
        recs["item_id"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
recs = recs.dropna()
recs = pl.from_pandas(recs)

In [ ]:
%%time

recs.write_parquet(config.cur_model_output_path + f'cur_als_df_val_preds_{config.num_recs}.parquet')

In [ ]:
recs.head()

In [ ]:
X_val_seq.head()

In [ ]:
X_val.filter(pl.col("user") == 14378816).head()

In [ ]:
recs = (
    recs.groupby(["index"])
    .agg(
        [
            pl.col("item_id").list(),
        ]
    )
    .sort("index")
    .to_pandas()
)

In [ ]:
recs["user"] = X_val_seq.to_pandas()["user"]
recs.head()

In [ ]:
%%time

recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=False, max_len=20))

In [ ]:
recs = recs.merge(X_val[["user", "target"]].to_pandas())

In [ ]:
recs.head()

In [ ]:
%%time

preds = recs["item_id"].tolist()
preds = [list(pred) for pred in preds]

targets = recs["target"].tolist()
targets = [list(target) for target in targets]

utils.recall_several_targets(preds, targets)

In [ ]:
stop

In [ ]:
X_val.to_pandas()[["user", "target"]]

In [ ]:
recs['target'] = 

In [ ]:
recs.filter(pl.col("index") == 1).head(20)

In [ ]:
stop

In [ ]:
%%time

rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)

data_to_predict = X_val_seq.to_pandas()[config.col_train_seq].tolist()
df_preds = rec_cooc.recommend(data_to_predict)

df_val_users = pl.from_pandas(X_val_seq.to_pandas()[[config.col_user]].reset_index())
df_val_preds = df_val_users.join(df_preds, on='index', how='inner')
df_val_preds = df_val_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_val_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_val_preds_{config.num_recs}.parquet')

In [ ]:
del data_to_predict, df_preds, df_val_users, df_val_preds, X_val_seq, X_val
gc.collect()

In [ ]:
del top_pairs
gc.collect()

# TEST

In [ ]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

In [ ]:
%%time

top_pairs = coocurence.train_cooc_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=20)

In [ ]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()
rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)
df_preds = rec_cooc.recommend(data_to_predict)

df_test_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].reset_index())
df_test_preds = df_test_users.join(df_preds, on='index', how='inner')
df_test_preds = df_test_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_test_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_test_preds_{config.num_recs}.parquet')